# Homework 9: Serverless Deep Learning

In this homework, we'll deploy the Straight vs Curly Hair Type model we trained in the previous homework.

Download the model files from here:

* https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx.data
* https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx


In [1]:
# !wget https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx.data
# !wget https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx

## Question 1

To be able to use this model, we need to know the name of the input and output nodes.

What's the name of the output?

* `output`
* `sigmoid`
* `softmax`
* `prediction`

In [2]:
import onnxruntime as ort

model_path = 'hair_classifier_v1.onnx'
session = ort.InferenceSession(model_path)

print("Input names:")
for i in session.get_inputs():
    print(i.name)

print("\nOutput names:")
for o in session.get_outputs():
    print(o.name)

Input names:
input

Output names:
output


**Answer:** `output`

## Question 2

Let's download and resize this image:

https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

Based on the previous homework, what should be the target size for the image?

* 64x64
* 128x128
* 200x200
* 256x256

In [3]:
print("Input shape:")
for i in session.get_inputs():
    print(i.name, i.shape)

Input shape:
input ['s77', 3, 200, 200]


**Answer:** `200x200`

## Question 3

Now we need to turn the image into numpy array and pre-process it.

> Tip: Check the previous homework. What was the pre-processing we did there?

After the pre-processing, what's the value in the first pixel, the R channel?

* -10.73
* -1.073
* 1.073
* 10.73

In [4]:
from io import BytesIO
from urllib import request
from PIL import Image
import numpy as np

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img

def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

url = 'https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg'
target_size = (200, 200)

img = download_image(url)
img_prep = prepare_image(img, target_size)

x = np.array(img_prep, dtype='float32')
x /= 255.0
mean = np.array([0.485, 0.456, 0.406], dtype='float32')
std = np.array([0.229, 0.224, 0.225], dtype='float32')
x = (x - mean) / std

print("First pixel R channel value:", x[0, 0, 0])

First pixel R channel value: -1.073294


**Answer:** `-1.073`

## Question 4

Now let's apply this model to this image. What's the output of the model?

* 0.09
* 0.49
* 0.69
* 0.89

In [5]:
X = np.array([x])
X = np.transpose(X, (0, 3, 1, 2))

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

pred = session.run([output_name], {input_name: X})[0]
print("Prediction:", pred[0][0])

Prediction: 0.091565885


**Answer:** `0.09`

## Question 5

Download the base image `agrigorev/model-2025-hairstyle:v1`.

So what's the size of this base image?

* 88 Mb
* 208 Mb
* 608 Mb
* 1208 Mb

In [10]:
!docker pull agrigorev/model-2025-hairstyle:v1
!docker images | grep agrigorev/model-2025-hairstyle

v1: Pulling from agrigorev/model-2025-hairstyle
Digest: sha256:9e43d5a5323f7f07688c0765d3c0137af66d0154af37833ed721d6b4de6df528
Status: Image is up to date for agrigorev/model-2025-hairstyle:v1
docker.io/agrigorev/model-2025-hairstyle:v1
agrigorev/model-2025-hairstyle:v1   4528ad1525d5        608MB             0B        


**Answer:** `608` (based on local docker console output)

## Question 6

Now let's extend this docker image, install all the required libraries and add the code for lambda.

Score this image: https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

What's the output from the model?

* -1.0
* -0.10
* 0.10
* 1.0

In [ ]:
# Build and run the docker container (see Dockerfile and lambda_function.py)
# python test_lambda_local.py

# docker build -t hairstyle-model .
# docker run -it --rm -p 8080:8080 hairstyle-model
# python test_lambda_local.py

**Answer:** `0.10` (based on local inference result 0.09)